In [ ]:
# ============================================================
# 🎵 MOOD-BASED MUSIC RECOMMENDATION CHATBOT
# AI-Based Text & Speech Analysis using NLP
# Google Colab - ONE CELL PROJECT
# ============================================================

# -----------------------------
# 1. INSTALL REQUIRED PACKAGES
# -----------------------------
!pip -q install gradio==5.44.1 faster-whisper

# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------
import re
import gradio as gr
from collections import Counter
from faster_whisper import WhisperModel

print("==========================================")
print("🎵 MOOD MUSIC CHATBOT")
print("==========================================")
print("Loading Speech-to-Text model...")

# -----------------------------
# 3. LOAD WHISPER MODEL
# -----------------------------
whisper_model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("✅ Speech-to-Text model loaded!")
print("Starting chatbot...\n")


# ============================================================
# 4. MOOD KEYWORDS
# ============================================================

MOOD_KEYWORDS = {

    "Happy": [
        "happy", "joy", "joyful", "excited", "great",
        "good", "awesome", "amazing", "fun", "wonderful",
        "cheerful", "smile", "celebrate", "celebration",
        "positive", "fantastic", "glad", "delighted"
    ],

    "Sad": [
        "sad", "cry", "crying", "depressed", "lonely",
        "alone", "heartbroken", "upset", "hurt", "pain",
        "lost", "miss", "missing", "tears", "down",
        "disappointed", "breakup"
    ],

    "Angry": [
        "angry", "anger", "mad", "furious", "irritated",
        "annoyed", "hate", "frustrated", "rage", "fight",
        "worst", "disgusted"
    ],

    "Stressed": [
        "stress", "stressed", "pressure", "tension",
        "exam", "exams", "deadline", "workload",
        "busy", "worried", "worry", "anxious",
        "anxiety", "panic", "overthinking",
        "problem", "problems", "focus"
    ],

    "Relaxed": [
        "calm", "relaxed", "peaceful", "peace",
        "chill", "comfortable", "quiet", "fresh",
        "relax", "meditation", "meditating",
        "nature", "sleepy"
    ],

    "Energetic": [
        "energy", "energetic", "active", "motivated",
        "motivation", "power", "powerful", "gym",
        "workout", "dance", "party", "running",
        "run", "productive", "confident"
    ],

    "Romantic": [
        "romantic", "romance", "crush", "love",
        "lover", "date", "dating", "relationship",
        "girlfriend", "boyfriend", "couple"
    ],

    "Tired": [
        "tired", "exhausted", "sleep", "sleepy",
        "fatigue", "drained", "rest", "no energy",
        "weak", "burnout", "burned out"
    ]
}


# ============================================================
# 5. MUSIC RECOMMENDATION DATABASE
# ============================================================

MUSIC = {

    "Happy": {
        "genres": [
            "Pop",
            "Feel-Good",
            "Dance"
        ],
        "songs": [
            "Can't Stop the Feeling! — Justin Timberlake",
            "Happy — Pharrell Williams",
            "Good as Hell — Lizzo",
            "On Top of the World — Imagine Dragons",
            "Best Day of My Life — American Authors"
        ],
        "advice":
            "Choose bright and upbeat music to maintain your positive energy."
    },

    "Sad": {
        "genres": [
            "Soft Pop",
            "Acoustic",
            "Melodic"
        ],
        "songs": [
            "Someone Like You — Adele",
            "Let Her Go — Passenger",
            "Photograph — Ed Sheeran",
            "Lovely — Billie Eilish & Khalid",
            "The Night We Met — Lord Huron"
        ],
        "advice":
            "Start with gentle music and slowly move toward hopeful songs."
    },

    "Angry": {
        "genres": [
            "Rock",
            "Alternative",
            "Power Pop"
        ],
        "songs": [
            "Believer — Imagine Dragons",
            "Numb — Linkin Park",
            "Centuries — Fall Out Boy",
            "Natural — Imagine Dragons",
            "Whatever It Takes — Imagine Dragons"
        ],
        "advice":
            "Energetic music can help you release tension before switching to calmer tracks."
    },

    "Stressed": {
        "genres": [
            "Lo-fi",
            "Ambient",
            "Soft Piano"
        ],
        "songs": [
            "Weightless — Marconi Union",
            "River Flows in You — Yiruma",
            "Comptine d'un autre été — Yann Tiersen",
            "A Moment Apart — ODESZA",
            "Sunset Lover — Petit Biscuit"
        ],
        "advice":
            "Try low-tempo instrumental music while taking a few slow breaths."
    },

    "Relaxed": {
        "genres": [
            "Ambient",
            "Chill",
            "Acoustic"
        ],
        "songs": [
            "Bloom — The Paper Kites",
            "Holocene — Bon Iver",
            "Sunset Lover — Petit Biscuit",
            "Better Together — Jack Johnson",
            "Yellow — Coldplay"
        ],
        "advice":
            "Keep the atmosphere soft with acoustic and ambient tracks."
    },

    "Energetic": {
        "genres": [
            "EDM",
            "Dance",
            "Workout"
        ],
        "songs": [
            "Don't Start Now — Dua Lipa",
            "Titanium — David Guetta ft. Sia",
            "Wake Me Up — Avicii",
            "On Top of the World — Imagine Dragons",
            "The Nights — Avicii"
        ],
        "advice":
            "Use high-energy beats for workouts, productivity and motivation."
    },

    "Romantic": {
        "genres": [
            "Romantic Pop",
            "R&B",
            "Acoustic"
        ],
        "songs": [
            "Perfect — Ed Sheeran",
            "All of Me — John Legend",
            "Until I Found You — Stephen Sanchez",
            "A Thousand Years — Christina Perri",
            "Lover — Taylor Swift"
        ],
        "advice":
            "Soft vocals and warm acoustic tracks fit a romantic mood."
    },

    "Tired": {
        "genres": [
            "Lo-fi",
            "Sleep",
            "Ambient"
        ],
        "songs": [
            "Weightless — Marconi Union",
            "Experience — Ludovico Einaudi",
            "Nuvole Bianche — Ludovico Einaudi",
            "Clair de Lune — Claude Debussy",
            "Comptine d'un autre été — Yann Tiersen"
        ],
        "advice":
            "Choose slow, low-volume music and give yourself time to rest."
    },

    "Neutral": {
        "genres": [
            "Lo-fi",
            "Pop",
            "Acoustic"
        ],
        "songs": [
            "As It Was — Harry Styles",
            "Golden — Harry Styles",
            "Until I Found You — Stephen Sanchez",
            "Photograph — Ed Sheeran",
            "Sunset Lover — Petit Biscuit"
        ],
        "advice":
            "Your mood seems balanced, so explore a comfortable mix of genres."
    }
}


# ============================================================
# 6. TEXT CLEANING
# ============================================================

def normalize(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 7. MOOD DETECTION
# ============================================================

def detect_mood(text):

    clean = normalize(text)

    if not clean:

        return (
            "Neutral",
            50,
            {},
            []
        )

    words = set(clean.split())

    scores = {}

    matched_words = {}

    # Calculate keyword score
    for mood, keywords in MOOD_KEYWORDS.items():

        hits = []

        for keyword in keywords:

            if " " in keyword:

                if keyword in clean:
                    hits.append(keyword)

            else:

                if keyword in words:
                    hits.append(keyword)

        scores[mood] = len(hits)

        matched_words[mood] = hits


    # Phrase-based score improvement
    phrase_boosts = {

        "Stressed": [
            "can't focus",
            "cannot focus",
            "too much",
            "exam tomorrow"
        ],

        "Sad": [
            "feel alone",
            "feeling alone",
            "miss you",
            "not okay"
        ],

        "Happy": [
            "very happy",
            "so happy",
            "feeling good"
        ],

        "Angry": [
            "so angry",
            "very angry"
        ],

        "Tired": [
            "so tired",
            "very tired"
        ]
    }


    for mood, phrases in phrase_boosts.items():

        for phrase in phrases:

            if phrase in clean:

                scores[mood] += 2

                matched_words[mood].append(
                    phrase
                )


    # Find highest mood
    best_mood = max(
        scores,
        key=scores.get
    )

    best_score = scores[best_mood]

    total_score = sum(
        scores.values()
    )


    # No keywords found
    if best_score == 0:

        return (
            "Neutral",
            55,
            scores,
            []
        )


    # Confidence calculation
    confidence = int(
        55 +
        (
            best_score /
            max(total_score, 1)
        ) * 45
    )

    confidence = min(
        confidence,
        98
    )


    return (
        best_mood,
        confidence,
        scores,
        matched_words[best_mood]
    )


# ============================================================
# 8. CHATBOT RESPONSE
# ============================================================

def create_response(
    user_text,
    mood,
    confidence
):

    music = MUSIC[mood]


    intro = {

        "Happy":
            "You sound happy and positive! 😊",

        "Sad":
            "It sounds like you may be feeling a little low. 💙",

        "Angry":
            "I can sense some frustration or anger in your message. 🔥",

        "Stressed":
            "It sounds like you are dealing with some stress or pressure. 🌿",

        "Relaxed":
            "You seem calm and relaxed right now. 🌸",

        "Energetic":
            "You sound energetic and motivated! ⚡",

        "Romantic":
            "Your message has a warm and romantic vibe. ❤️",

        "Tired":
            "You sound tired and may need some rest. 😴",

        "Neutral":
            "Your message has a fairly balanced mood. 🙂"
    }[mood]


    response = f"""
# 🎵 Mood Analysis Result

### Detected Mood
## **{mood}**

**Confidence:** `{confidence}%`

---

### 💬 Chatbot

{intro}

**My recommendation:**

{music["advice"]}

---

### 🎧 Recommended Genres

"""

    for genre in music["genres"]:

        response += f"- 🎶 **{genre}**\n"


    response += """

---

### 🎵 Recommended Songs

"""


    for song in music["songs"]:

        response += f"- 🎵 {song}\n"


    response += """

---

### 💡 TSA Analysis

The chatbot analyzed your message using:

- Text preprocessing
- Keyword matching
- Mood classification
- NLP-based analysis
- Recommendation system

"""

    return response


# ============================================================
# 9. TEXT CHAT FUNCTION
# ============================================================

def analyze_text(
    text,
    history
):

    if not text or not text.strip():

        return (
            "⚠️ Please type how you are feeling first.",
            "",
            history
        )


    mood, confidence, scores, matched = detect_mood(
        text
    )


    response = create_response(
        text,
        mood,
        confidence
    )


    if history is None:

        history = []


    history.append({

        "text": text,

        "mood": mood,

        "confidence": confidence
    })


    history_display = ""

    for item in history[-10:]:

        history_display += f"""
### 👤 You
{item["text"]}

### 🤖 Bot
Mood: **{item["mood"]}**
Confidence: **{item["confidence"]}%**

---
"""


    return (
        response,
        history_display,
        history
    )


# ============================================================
# 10. SPEECH TO TEXT
# ============================================================

def speech_to_text(audio):

    if audio is None:

        return ""


    try:

        segments, info = whisper_model.transcribe(
            audio,
            beam_size=1
        )


        text = " ".join(
            segment.text.strip()
            for segment in segments
        )


        return text.strip()


    except Exception as error:

        return f"Speech recognition error: {error}"


# ============================================================
# 11. VOICE ANALYSIS
# ============================================================

def analyze_voice(
    audio,
    history
):

    text = speech_to_text(
        audio
    )


    if not text:

        return (
            "⚠️ No speech detected. Please try again.",
            "",
            history,
            ""
        )


    response, history_display, history = analyze_text(
        text,
        history
    )


    return (
        response,
        history_display,
        history,
        text
    )


# ============================================================
# 12. FINAL REPORT
# ============================================================

def generate_report(history):

    if not history:

        return """
# 📊 Mood Analysis Report

No analysis has been performed yet.

Please use the Text Chat or Voice Analysis tab first.
"""


    moods = [
        item["mood"]
        for item in history
    ]


    counts = Counter(
        moods
    )


    most_common_mood = counts.most_common(1)[0][0]


    report = f"""
# 📊 Final Mood Analysis Report

## 📌 Summary

**Total Interactions:** {len(history)}

**Most Detected Mood:**
# {most_common_mood}

---

## 📈 Mood Distribution

"""


    for mood, count in counts.most_common():

        percentage = int(
            count /
            len(history)
            * 100
        )

        report += f"""
### {mood}

`{count}` interaction(s) — **{percentage}%**

"""


    report += """
---

## 📝 Recent Analysis

"""


    for index, item in enumerate(
        history[-10:],
        1
    ):

        report += f"""
**{index}. {item["mood"]}**

User: {item["text"]}

Confidence: {item["confidence"]}%

---
"""


    return report


# ============================================================
# 13. CLEAR FUNCTION
# ============================================================

def clear_all():

    return (
        "",
        "",
        "",
        [],
        None,
        ""
    )


# ============================================================
# 14. CUSTOM CSS
# ============================================================

custom_css = """

.gradio-container {

    max-width: 1100px !important;

    margin: auto !important;

}


.hero {

    text-align: center;

    padding: 30px;

    border-radius: 25px;

    background:
    linear-gradient(
        135deg,
        #5b21b6,
        #db2777,
        #f59e0b
    );

    color: white;

    margin-bottom: 20px;

}


.hero h1 {

    font-size: 36px;

}


.feature-card {

    border-radius: 20px;

}


footer {

    display: none !important;

}

"""


# ============================================================
# 15. BUILD GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="Mood-Based Music Recommendation Chatbot"
) as app:


    # ------------------------------------------
    # HERO SECTION
    # ------------------------------------------

    gr.HTML("""

    <div class="hero">

        <h1>
        🎵 Mood-Based Music Recommendation Chatbot
        </h1>

        <h3>
        AI-Based Text & Speech Analysis using NLP
        </h3>

        <p>
        Tell me how you feel and I will detect your
        mood and recommend suitable music.
        </p>

    </div>

    """)


    # ------------------------------------------
    # FEATURES
    # ------------------------------------------

    gr.Markdown("""
## ✨ Features

| Feature | Description |
|---|---|
| 💬 Text Analysis | Analyze your typed message |
| 🎙️ Voice Analysis | Convert speech into text |
| 🧠 NLP | Detect emotional keywords |
| 😊 Mood Detection | Identify your current mood |
| 🎧 Recommendation | Suggest suitable music |
| 📊 Report | Show mood history |
""")


    # Shared state
    state = gr.State([])


    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():


        # ====================================================
        # TEXT CHAT
        # ====================================================

        with gr.Tab("💬 Text Chat"):

            gr.Markdown("""
### 💬 Tell me how you feel

Example:

> I am very stressed because I have an exam tomorrow.
""")

            text_input = gr.Textbox(

                label="Your Message",

                placeholder=
                "Type how you are feeling...",

                lines=5
            )


            text_button = gr.Button(

                "🔍 Analyze Mood & Recommend Music",

                variant="primary"
            )


            text_result = gr.Markdown()


            gr.Markdown(
                "## 💬 Conversation History"
            )


            text_history = gr.Markdown()


            text_button.click(

                analyze_text,

                inputs=[
                    text_input,
                    state
                ],

                outputs=[
                    text_result,
                    text_history,
                    state
                ]
            )


        # ====================================================
        # VOICE ANALYSIS
        # ====================================================

        with gr.Tab("🎙️ Voice Analysis"):

            gr.Markdown("""
### 🎙️ Speak to the chatbot

Record your voice and the system will:

**Voice → Speech-to-Text → NLP → Mood Detection → Music Recommendation**
""")


            audio_input = gr.Audio(

                sources=[
                    "microphone",
                    "upload"
                ],

                type="filepath",

                label="🎙️ Record or Upload Voice"
            )


            voice_button = gr.Button(

                "🎙️ Analyze Voice",

                variant="primary"
            )


            transcript = gr.Textbox(

                label="🗣️ Speech-to-Text Result",

                lines=4
            )


            voice_result = gr.Markdown()


            gr.Markdown(
                "## 💬 Voice Conversation History"
            )


            voice_history = gr.Markdown()


            voice_button.click(

                analyze_voice,

                inputs=[
                    audio_input,
                    state
                ],

                outputs=[
                    voice_result,
                    voice_history,
                    state,
                    transcript
                ]
            )


        # ====================================================
        # FINAL REPORT
        # ====================================================

        with gr.Tab("📊 Final Report"):

            gr.Markdown("""
## 📊 Mood Analysis Report

Generate a report after using the chatbot.
""")


            report_button = gr.Button(

                "📊 Generate Final Report",

                variant="primary"
            )


            report_output = gr.Markdown()


            report_button.click(

                generate_report,

                inputs=state,

                outputs=report_output
            )


    # ========================================================
    # CLEAR BUTTON
    # ========================================================

    gr.Markdown("---")


    clear_button = gr.Button(
        "🗑️ Clear All",
        variant="stop"
    )


    clear_button.click(

        clear_all,

        outputs=[
            text_input,
            text_result,
            text_history,
            state,
            audio_input,
            transcript
        ]
    )


    # ========================================================
    # PROJECT INFORMATION
    # ========================================================

    gr.Markdown("""
---

# 🎓 Project Information

### Project Title

**AI-Based Mood-Based Music Recommendation Chatbot using Text and Speech Analysis**

### Technologies

- Python
- NLP
- Faster-Whisper
- Gradio
- Google Colab

### TSA Concepts

- Text Analysis
- Speech Analysis
- Speech-to-Text
- NLP
- Emotion Detection
- Keyword Extraction
- Chatbot
- Recommendation System
- Confidence Scoring

---

### ⚠️ Note

This is an educational NLP project. Mood detection is an approximate
text-analysis result and is not a medical or psychological diagnosis.
""")


# ============================================================
# 16. LAUNCH PUBLIC URL
# ============================================================

print("\n==========================================")
print("🚀 LAUNCHING CHATBOT")
print("==========================================")
print()
print("⏳ Please wait...")
print()
print("Your PUBLIC URL will appear below.")
print("Look for:")
print()
print("👉 Running on public URL: https://xxxx.gradio.live")
print()

app.launch(
    share=True,
    debug=True
)

🎵 MOOD MUSIC CHATBOT
Loading Speech-to-Text model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

✅ Speech-to-Text model loaded!
Starting chatbot...


🚀 LAUNCHING CHATBOT

⏳ Please wait...

Your PUBLIC URL will appear below.
Look for:

👉 Running on public URL: https://xxxx.gradio.live

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d84621cc00da71e428.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
